# Train CNN Explainer TinyVGG Models in PyTorch

This Colab notebook trains PyTorch equivalents of the existing CNN Explainer TensorFlow.js models:

- `tiny-vgg-7` -> `backend/models/tiny_vgg_7.pt`
- `tiny-vgg-12` -> `backend/models/tiny_vgg_12.pt`
- `tiny-vgg-17` -> `backend/models/tiny_vgg_17.pt`

The saved checkpoints use `state_dict` format with an `architectureKey`, so the Flask backend can load them safely without relying on pickled notebook classes.

## 1. Clone Repo and Prepare Data

Use a GPU runtime in Colab: `Runtime -> Change runtime type -> T4 GPU` or better.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import time
import zipfile

REPO_URL = "https://github.com/batis1/CNN-Visualizer.git"
BRANCH = "day-01-overview"
REPO_DIR = Path("/content/CNN-Visualizer")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
DATA_ZIP = REPO_DIR / "tiny-vgg" / "data.zip"
DATA_DIR = REPO_DIR / "tiny-vgg" / "data"

if not DATA_DIR.exists():
    with zipfile.ZipFile(DATA_ZIP, "r") as archive:
        archive.extractall(REPO_DIR / "tiny-vgg")

print("repo:", REPO_DIR)
print("data:", DATA_DIR)


## 2. Imports and Dataset

In [ ]:
import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms

CLASS_LABELS = [
    "lifeboat",
    "ladybug",
    "pizza",
    "bell pepper",
    "school bus",
    "koala",
    "espresso",
    "red panda",
    "orange",
    "sport car",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

image_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])


class TinyVggDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = list(paths)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = Image.open(self.paths[index]).convert("RGB")
        image = self.transform(image)
        label = int(self.labels[index])
        return image, label


def load_class_dicts(data_dir):
    with open(data_dir / "class_dict_10.json", "r", encoding="utf-8") as file:
        train_class_dict = json.load(file)
    with open(data_dir / "val_class_dict_10.json", "r", encoding="utf-8") as file:
        val_class_dict = json.load(file)
    return train_class_dict, val_class_dict


def build_datasets(data_dir):
    train_class_dict, val_class_dict = load_class_dicts(data_dir)

    train_paths = sorted((data_dir / "class_10_train").glob("*/images/*.JPEG"))
    train_labels = [train_class_dict[path.parent.parent.name]["index"] for path in train_paths]

    val_paths = sorted((data_dir / "class_10_val" / "val_images").glob("*.JPEG"))
    val_labels = [val_class_dict[path.name]["index"] for path in val_paths]

    test_paths = sorted((data_dir / "class_10_val" / "test_images").glob("*.JPEG"))
    test_labels = [val_class_dict[path.name]["index"] for path in test_paths]

    return {
        "train": TinyVggDataset(train_paths, train_labels, image_transform),
        "val": TinyVggDataset(val_paths, val_labels, image_transform),
        "test": TinyVggDataset(test_paths, test_labels, image_transform),
    }


datasets = build_datasets(DATA_DIR)
print({name: len(dataset) for name, dataset in datasets.items()})


## 3. PyTorch Architectures

The final layer returns logits. The backend applies softmax when it builds `/api/explain` JSON.

In [ ]:
class TinyVgg7(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_1_1 = nn.Conv2d(3, 6, kernel_size=3)
        self.relu_1_1 = nn.ReLU()
        self.conv_1_2 = nn.Conv2d(6, 6, kernel_size=3)
        self.sigmoid_1_1 = nn.Sigmoid()
        self.avg_pool_1 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dense_1 = nn.Linear(5400, 64)
        self.relu_dense_1 = nn.ReLU()
        self.dense_2 = nn.Linear(64, 32)
        self.relu_dense_2 = nn.ReLU()
        self.output = nn.Linear(32, 10)

    def forward(self, x):
        x = self.conv_1_1(x)
        x = self.relu_1_1(x)
        x = self.conv_1_2(x)
        x = self.sigmoid_1_1(x)
        x = self.avg_pool_1(x)
        x = self.flatten(x)
        x = self.dense_1(x)
        x = self.relu_dense_1(x)
        x = self.dense_2(x)
        x = self.relu_dense_2(x)
        return self.output(x)


class TinyVgg12(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_1_1 = nn.Conv2d(3, 10, kernel_size=3)
        self.relu_1_1 = nn.ReLU()
        self.conv_1_2 = nn.Conv2d(10, 10, kernel_size=3)
        self.relu_1_2 = nn.ReLU()
        self.avg_pool_1 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv_2_1 = nn.Conv2d(10, 10, kernel_size=3)
        self.relu_2_1 = nn.ReLU()
        self.conv_2_2 = nn.Conv2d(10, 10, kernel_size=3)
        self.relu_2_2 = nn.ReLU()
        self.max_pool_2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dense_1 = nn.Linear(1690, 64)
        self.relu_dense_1 = nn.ReLU()
        self.dense_2 = nn.Linear(64, 32)
        self.relu_dense_2 = nn.ReLU()
        self.output = nn.Linear(32, 10)

    def forward(self, x):
        x = self.conv_1_1(x)
        x = self.relu_1_1(x)
        x = self.conv_1_2(x)
        x = self.relu_1_2(x)
        x = self.avg_pool_1(x)
        x = self.conv_2_1(x)
        x = self.relu_2_1(x)
        x = self.conv_2_2(x)
        x = self.relu_2_2(x)
        x = self.max_pool_2(x)
        x = self.flatten(x)
        x = self.dense_1(x)
        x = self.relu_dense_1(x)
        x = self.dense_2(x)
        x = self.relu_dense_2(x)
        return self.output(x)


class TinyVgg17(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_1_1 = nn.Conv2d(3, 14, kernel_size=3)
        self.relu_1_1 = nn.ReLU()
        self.conv_1_2 = nn.Conv2d(14, 14, kernel_size=3)
        self.relu_1_2 = nn.ReLU()
        self.avg_pool_1 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv_2_1 = nn.Conv2d(14, 14, kernel_size=3)
        self.relu_2_1 = nn.ReLU()
        self.conv_2_2 = nn.Conv2d(14, 14, kernel_size=3)
        self.relu_2_2 = nn.ReLU()
        self.max_pool_2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv_3_1 = nn.Conv2d(14, 14, kernel_size=3)
        self.relu_3_1 = nn.ReLU()
        self.conv_3_2 = nn.Conv2d(14, 14, kernel_size=3)
        self.relu_3_2 = nn.ReLU()
        self.avg_pool_3 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dense_1 = nn.Linear(224, 64)
        self.relu_dense_1 = nn.ReLU()
        self.dense_2 = nn.Linear(64, 32)
        self.relu_dense_2 = nn.ReLU()
        self.output = nn.Linear(32, 10)

    def forward(self, x):
        x = self.conv_1_1(x)
        x = self.relu_1_1(x)
        x = self.conv_1_2(x)
        x = self.relu_1_2(x)
        x = self.avg_pool_1(x)
        x = self.conv_2_1(x)
        x = self.relu_2_1(x)
        x = self.conv_2_2(x)
        x = self.relu_2_2(x)
        x = self.max_pool_2(x)
        x = self.conv_3_1(x)
        x = self.relu_3_1(x)
        x = self.conv_3_2(x)
        x = self.relu_3_2(x)
        x = self.avg_pool_3(x)
        x = self.flatten(x)
        x = self.dense_1(x)
        x = self.relu_dense_1(x)
        x = self.dense_2(x)
        x = self.relu_dense_2(x)
        return self.output(x)


MODEL_BUILDERS = {
    "tiny-vgg-7": TinyVgg7,
    "tiny-vgg-12": TinyVgg12,
    "tiny-vgg-17": TinyVgg17,
}

MODEL_FILES = {
    "tiny-vgg-7": "tiny_vgg_7.pt",
    "tiny-vgg-12": "tiny_vgg_12.pt",
    "tiny-vgg-17": "tiny_vgg_17.pt",
}


## 4. Training Configuration

For a first smoke run, set `QUICK_SMOKE_TEST = True` and train only one model. For final checkpoints, keep it `False`.

In [ ]:
MODEL_KEYS = ["tiny-vgg-7", "tiny-vgg-12", "tiny-vgg-17"]
QUICK_SMOKE_TEST = False

BATCH_SIZE = 64
MAX_EPOCHS = 80
PATIENCE = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

if QUICK_SMOKE_TEST:
    MODEL_KEYS = ["tiny-vgg-7"]
    MAX_EPOCHS = 2
    PATIENCE = 1

OUTPUT_DIR = REPO_DIR / "backend" / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("models:", MODEL_KEYS)
print("output:", OUTPUT_DIR)


## 5. Train and Save Checkpoints

In [ ]:
def maybe_subset(dataset, max_items):
    if not QUICK_SMOKE_TEST or len(dataset) <= max_items:
        return dataset
    generator = torch.Generator().manual_seed(42)
    indices = torch.randperm(len(dataset), generator=generator)[:max_items].tolist()
    return Subset(dataset, indices)


def make_loaders():
    train_dataset = maybe_subset(datasets["train"], 512)
    val_dataset = maybe_subset(datasets["val"], 128)
    test_dataset = maybe_subset(datasets["test"], 128)
    num_workers = 2 if DEVICE.type == "cuda" else 0

    return {
        "train": DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=DEVICE.type == "cuda"),
        "val": DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=DEVICE.type == "cuda"),
        "test": DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=DEVICE.type == "cuda"),
    }


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(images)
            loss = criterion(logits, labels)
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_count += batch_size

    return total_loss / total_count, total_correct / total_count


def save_checkpoint(model, architecture_key, path, metrics):
    torch.save(
        {
            "format": "cnn-explainer-tiny-vgg-state-dict-v1",
            "architectureKey": architecture_key,
            "inputShape": [3, 64, 64],
            "classLabels": CLASS_LABELS,
            "metrics": metrics,
            "model_state_dict": model.cpu().state_dict(),
        },
        path,
    )
    model.to(DEVICE)


def train_one_model(architecture_key):
    loaders = make_loaders()
    model = MODEL_BUILDERS[architecture_key]().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_val_loss = float("inf")
    best_metrics = None
    best_state = None
    bad_epochs = 0
    start_time = time.time()

    print(f"\n=== Training {architecture_key} ===")
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_count = 0

        for images, labels in loaders["train"]:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            train_loss_sum += loss.item() * batch_size
            train_correct += (logits.argmax(dim=1) == labels).sum().item()
            train_count += batch_size

        train_loss = train_loss_sum / train_count
        train_acc = train_correct / train_count
        val_loss, val_acc = evaluate(model, loaders["val"], criterion)

        print(
            f"epoch {epoch:03d} | train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            best_metrics = {
                "bestEpoch": epoch,
                "trainLoss": train_loss,
                "trainAccuracy": train_acc,
                "validationLoss": val_loss,
                "validationAccuracy": val_acc,
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= PATIENCE:
            print(f"early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    test_loss, test_acc = evaluate(model, loaders["test"], criterion)
    best_metrics.update(
        {
            "testLoss": test_loss,
            "testAccuracy": test_acc,
            "elapsedMinutes": (time.time() - start_time) / 60,
        }
    )

    output_path = OUTPUT_DIR / MODEL_FILES[architecture_key]
    save_checkpoint(model, architecture_key, output_path, best_metrics)
    print(f"saved {output_path}")
    print(best_metrics)
    return {"architectureKey": architecture_key, "path": str(output_path), **best_metrics}


training_results = [train_one_model(model_key) for model_key in MODEL_KEYS]
training_results


## 6. Package Outputs

In [ ]:
summary_path = OUTPUT_DIR / "tiny_vgg_training_results.json"
summary_path.write_text(json.dumps(training_results, indent=2), encoding="utf-8")

zip_base = OUTPUT_DIR / "tiny_vgg_pytorch_models"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR))
print("summary:", summary_path)
print("zip:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print("Download manually from:", zip_path)
    print(exc)
